# F1-Race – your first race

Three drivers, one flying lap at Spielberg, a classification and a replay you can scrub through.
Each driver is a way to plan the lap; the plan is then **driven** by the L0 tracker on the
kinematic car in the simulator (see the [README](https://github.com/yuanche-auto/F1-Race#readme)).

| Driver | How it plans the lap |
|---|---|
| **OCP** | minimum-time optimal control – line and speed optimised together |
| **MCV** | the minimum-curvature line, then the fastest speed profile on it |
| **CTR** | the centre line with the same speed profile |

Run the cells from top to bottom: in Colab the first one fetches the code, locally (Jupyter in a
checkout, after `pip install -e .`) it only moves to the repository root. The first race takes
about a minute (the plans); the plans are cached, so the next races take seconds.

In [ ]:
import os, sys

if "google.colab" in sys.modules:                 # Colab: fetch the code and install it
    if not os.path.exists("F1-Race"):
        !git clone -q https://github.com/yuanche-auto/F1-Race.git
    %cd F1-Race
    !pip install -q -e .
elif os.path.basename(os.getcwd()) == "notebooks":  # a local checkout: work from its root
    os.chdir("..")

sys.path[:0] = ["src", "scripts"]
print(os.getcwd())

## 1 · The race

In [ ]:
from fetch_tracks import fetch
from f1race.race import RaceConfig, race, report, write_replay

fetch("Spielberg")                                # the TUM racetrack-database, pinned and SHA-256 checked
result = race(RaceConfig(track="Spielberg"))      # grip 0.97: the plans use 97 % of the friction ellipse
print()
print(report(result))

## 2 · The replay

Play, pause, scrub, change the speed, follow the leader.

In [ ]:
import html
from IPython.display import HTML


def show(res, name):
    page = write_replay(res, f"results/race/{name}.html").read_text(encoding="utf-8")
    return HTML(f'<div><iframe srcdoc="{html.escape(page)}" style="width:100%;height:820px;border:0"></iframe></div>')


show(result, "notebook_first_race")

## 3 · Push to the limit

With `grip=1.0` the plans use all of the grip and leave the tracker no room for its errors.
Who still finishes?

In [ ]:
limit = race(RaceConfig(track="Spielberg", grip=1.0))
print()
print(report(limit))
show(limit, "notebook_grip_1")

## Make it your race

* `RaceConfig(drivers=("ocp", "centre"))` – pick the field
* `RaceConfig(track="Monza")` – another circuit (`fetch("Monza")` first)
* `RaceConfig(margin=0.1)` – plan closer to the edges

From here the [layers](https://github.com/yuanche-auto/F1-Race#the-layers) make it harder: a car that
slides (L1), cars that differ from the model (L2), random disturbances (L3), …